# Replication FP16 gate - gemma-2-2b-it

Evaluates **google/gemma-2-2b-it** at **FP16 only**, on the same frozen 900-item
BELEBELE manifest, the same prompt template and the same `letter_logit` scoring
that produced P0.

**This notebook does not run INT8 or NF4.** A model at chance in a language
cannot show quantization degradation, so the floor gate in
`docs/H5_PREREGISTRATION.md` is applied to FP16 first. Bring the results back
before any quantized cell is authorised.

P0 is untouched by this notebook. `gemma-2-2b-it` is resolved from the
`replication_models` config key, which sits outside the frozen P0 subtree; the
freeze gate below proves that.

**Settings: Accelerator `GPU T4 x2`, Internet `ON`.**

In [ ]:
# 1. Get the code.
REPO_URL = "https://github.com/fairuz-anadi/quantization.git"
REF      = "main"          # branch, tag or commit SHA -- all three work

import os, subprocess, sys
SRC = "/kaggle/working/quantlang"
if not os.path.exists(SRC):
    subprocess.run(["git", "clone", REPO_URL, SRC], check=True)
    subprocess.run(["git", "-C", SRC, "checkout", "--quiet", REF], check=True)
print(subprocess.run(["git", "-C", SRC, "rev-parse", "HEAD"],
                     capture_output=True, text=True).stdout.strip())
os.chdir(SRC); sys.path.insert(0, SRC)

In [ ]:
# 1b. Hugging Face auth, for gated repositories.
#
# A gated model (Gemma, Llama) will NOT download without an accepted licence and
# a token. This reads it from Kaggle Secrets and reports plainly, so a missing
# token surfaces here rather than as a confusing 401 six minutes later.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as exc:
    print(f"no HF_TOKEN secret ({type(exc).__name__}). Fine for an open model; "
          f"a gated one fails at the preflight below with a clear message.")

In [ ]:
# 2. Dependencies. Kaggle's torch is CUDA-matched -- never reinstall it.
!pip install -q -U "transformers>=4.45" "bitsandbytes>=0.43" "peft>=0.13" accelerate datasets pyyaml
!pip uninstall -q -y torchao

In [ ]:
# 3. Environment probe.
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/probe_env.py", "--outdir", "/kaggle/working")

## P0 is still frozen

This is the check that matters for a replication run: adding a model must not
have disturbed the results P0 already produced. `replication_models` is a new
top-level key precisely so this passes.

In [ ]:
# 4. Contracts.
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/freeze_p0.py")
gate("-m", "pytest", "-q")

## Tokenizer pre-flight

Decided from the tokenizer alone, before any weights download. `letter_logit`
reads the logit of `" A"`..`" D"` and requires each to be exactly one distinct
token; a model that fails this cannot be scored by the P0 procedure at all, and
the answer is a different model -- never a different scoring method, which would
make the numbers incomparable with P0.

Also reports tokens per item per language on the frozen items, which is the
quantity H5 is about.

In [ ]:
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/probe_model_compat.py", "--hf-id", "google/gemma-2-2b-it", "--revision", "299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8", "--outdir", "/kaggle/working/rep")

## FP16 across all five languages

Same evaluator, same frozen manifest, same scoring as P0. `--model-alias
gemma-2-2b-it` resolves from `replication_models`; P0's own selection path is
untouched.

Roughly 35-50 minutes -- Sinhala dominates, at about 7x the tokens per item of
the other four under this tokenizer.

In [ ]:
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/run_eval.py", "--precision", "fp16",
     "--langs", 'eng_Latn', 'ben_Beng', 'sin_Sinh', 'asm_Beng', 'npi_Deva',
     "--model-alias", "gemma-2-2b-it",
     "--outdir", "/kaggle/working/rep/results", "--tag", "repfp16")

## The FP16 gate report

The floor gate is fixed in `docs/H5_PREREGISTRATION.md`: a language is
interpretable for this model when its **Wilson 95% lower bound exceeds 0.30**,
against a four-option chance level of 0.25. It is applied uniformly. A floored
language is reported as floored, not dropped.

In [ ]:
import glob, json, math
rows = []
for p in sorted(glob.glob("/kaggle/working/rep/results/*.meta.json")):
    m = json.load(open(p, encoding="utf-8"))
    n, k = m["n_items"], m["n_correct"]
    acc = k / n
    z = 1.959963985
    den = 1 + z*z/n
    centre = (acc + z*z/(2*n)) / den
    half = z*math.sqrt(acc*(1-acc)/n + z*z/(4*n*n)) / den
    lo, hi = centre - half, centre + half
    rows.append(dict(lang=m["lang"], n=n, correct=k, acc=acc, lo=lo, hi=hi,
                     tokens=m["median_input_tokens"], trunc=m["n_truncated"],
                     alias=m["model_alias"], arm=m["arm"],
                     passes=lo > 0.30))

print(f"model: {rows[0]['alias'] if rows else '?'}   chance = 0.25   "
      f"gate: Wilson lower bound > 0.30\n")
print(f"{'lang':10}{'acc':>8}{'95% CI':>18}{'items':>8}{'tok/item':>10}"
      f"{'trunc':>7}   gate")
for r in sorted(rows, key=lambda r: r["tokens"]):
    print(f"{r['lang']:10}{r['acc']:8.4f}"
          f"{'[' + format(r['lo'], '.3f') + ', ' + format(r['hi'], '.3f') + ']':>18}"
          f"{r['n']:8d}{r['tokens']:10.0f}{r['trunc']:7d}   "
          f"{'PASS' if r['passes'] else 'FLOORED'}")

bad = [r for r in rows if not r["passes"]]
print()
if bad:
    print(f"FLOORED: {[r['lang'] for r in bad]} -- no degradation claim will be "
          f"made for these cells. They are reported as floored, not omitted.")
else:
    print("All five languages clear the floor gate.")
assert all(r["arm"] == "base" for r in rows), "a replication cell must be a base cell"
assert all(r["n"] == 900 for r in rows), "every cell must be the full 900 items"
json.dump(rows, open("/kaggle/working/rep/fp16_gate.json", "w"), indent=2)
print("\nwrote /kaggle/working/rep/fp16_gate.json")
print("\nSTOP HERE. Bring these numbers back before any INT8 or NF4 cell is run.")

In [ ]:
import os, shutil
KEEP = "/kaggle/working/rep_gemma-2-2b-it_keep"
os.makedirs(KEEP, exist_ok=True)
shutil.copytree("/kaggle/working/rep/results", f"{KEEP}/results", dirs_exist_ok=True)
for f in ["fp16_gate.json"]:
    if os.path.exists(f"/kaggle/working/rep/{f}"):
        shutil.copy(f"/kaggle/working/rep/{f}", KEEP)
for f in os.listdir("/kaggle/working/rep"):
    if f.startswith("model_compat_"):
        shutil.copy(f"/kaggle/working/rep/{f}", KEEP)
shutil.make_archive("/kaggle/working/rep_gemma-2-2b-it", "zip", KEEP)
print(sorted(os.listdir("/kaggle/working")))